In [ ]:
from transformers import EarlyStoppingCallback  # Early stopping callback
from transformers import T5Tokenizer, T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset, Dataset
import warnings
import pandas as pd
import random
import numpy as np
import torch
from transformers import set_seed

# Set random seed
def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Make cudnn use deterministic algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

# Load custom CSV-format fine-tuning data
def load_custom_dataset(csv_path):
    df = pd.read_csv(csv_path)
    # Ensure the column names are ['Caption', 'SMILES']
    return Dataset.from_pandas(df)

# Data preprocessing, tokenize
def preprocess_function(examples, tokenizer, max_source_length=128, max_target_length=128):
    inputs = examples['Caption']
    targets = examples['SMILES']
    model_inputs = tokenizer(inputs, max_length=max_source_length, truncation=True, padding="max_length")

    labels = tokenizer(targets, max_length=max_target_length, truncation=True, padding="max_length")

    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in label] 
        for label in labels["input_ids"]
    ]
    
    model_inputs["labels"] = labels["input_ids"]
    
    return model_inputs

def main():
    # Paths for model and tokenizer
    model_name_or_path = R"YOUR_MODEL_PATH/molt5small_cp2sm"  # Local MolT5-small directory
    train_file = R"Dataset/MolGen_Train.csv"  # Training data path
    val_file = R"Dataset/MolGen_Evl.csv"  # Validation data path

    tokenizer = T5Tokenizer.from_pretrained(model_name_or_path)
    model = T5ForConditionalGeneration.from_pretrained(model_name_or_path)

    # Load training dataset
    raw_train_datasets = load_custom_dataset(train_file)
    tokenized_train_datasets = raw_train_datasets.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    # Load validation dataset
    raw_val_datasets = load_custom_dataset(val_file)
    tokenized_val_datasets = raw_val_datasets.map(lambda x: preprocess_function(x, tokenizer), batched=True)

    # Training parameters
    training_args = Seq2SeqTrainingArguments(
        output_dir="CheckPoint/SingMolT5_small",
        evaluation_strategy="epoch",  # Set to "epoch" to evaluate at the end of each epoch
        learning_rate=1e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=8,
        weight_decay=0.01,
        save_total_limit=3,
        num_train_epochs=50, # Up to 50 epochs
        predict_with_generate=True,
        logging_dir="CheckPoint/SingMolT5_small",
        logging_steps=10,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",  # Use validation loss to determine early stopping
        greater_is_better=False,            # Smaller loss is better
        report_to="none"
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train_datasets,
        eval_dataset=tokenized_val_datasets,
        tokenizer=tokenizer,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.05)]  # Early stopping: patience=3
    )

    # Start training
    trainer.train()

if __name__ == "__main__":
    warnings.filterwarnings("ignore", category=FutureWarning, module="accelerate") # ignore FutureWarning to improve output
    set_random_seed(42)
    main()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Map: 100%|██████████| 33/33 [00:00<00:00, 2153.23 examples/s]


Epoch,Training Loss,Validation Loss
1,7.541700,2.226257
2,5.887500,1.882142
3,4.655400,1.611059
4,4.187800,1.369608
5,3.305400,1.124361
6,2.607100,0.972677
7,2.550400,0.849030
8,2.100200,0.755820
9,2.042000,0.696303
10,1.686200,0.665750


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


# Notice
This warning "There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'] "occurs because the checkpoint stores the token embeddings under shared.weight (the T5 convention of using a single shared embedding). In other words, model.safetensors only contains shared.weight, so the loader reports those two encoder/decoder keys as missing. This is not actually an issue for us because we only use the shared embedding and never rely on encoder.embed_tokens.weight or decoder.embed_tokens.weight separately. For more context see this Hugging Face discussion: https://github.com/huggingface/transformers/issues/27972